In [29]:
import os
import json
import math
import time
import random
import shutil
import hashlib
import platform
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    log_loss,
    top_k_accuracy_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import models
from torchvision.models import EfficientNet_B0_Weights

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 7
NUM_CLASSES = 19
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = min(4, os.cpu_count() or 2)
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
ECE_BINS = 15

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [30]:
INPUT_ROOT = Path("/kaggle/input/datasets/katakuricharlotte")
CROP_ROOT = INPUT_ROOT / "crop-manifestfiles"
WORK_ROOT = Path("/kaggle/working/notebook12_outputs")

MANIFEST_ROOT = CROP_ROOT / "split_outputs" / "split_outputs"
EDA_ROOT = CROP_ROOT / "Eda-tables" / "Eda-tables"

OUTPUTS = {
    "root": WORK_ROOT,
    "audit": WORK_ROOT / "01_duplicate_audit",
    "manifests": WORK_ROOT / "02_audited_manifests",
    "training": WORK_ROOT / "03_training",
    "evaluation": WORK_ROOT / "04_evaluation",
    "figures": WORK_ROOT / "05_figures",
    "reports": WORK_ROOT / "06_reports",
    "checkpoints": WORK_ROOT / "checkpoints",
}

for folder in OUTPUTS.values():
    folder.mkdir(parents=True, exist_ok=True)

PATHS = {
    "main19_train": MANIFEST_ROOT / "manifests" / "main19" / "main19_train.csv",
    "main19_calibration": MANIFEST_ROOT / "manifests" / "main19" / "main19_calibration.csv",
    "main19_internal_test": MANIFEST_ROOT / "manifests" / "main19" / "main19_internal_test.csv",
    "review_progress": MANIFEST_ROOT / "tables" / "near_duplicate_manual_review_progress.csv",
    "review_decisions": MANIFEST_ROOT / "tables" / "near_duplicate_review_decisions.csv",
    "excluded_confirmed_near_dup": MANIFEST_ROOT / "tables" / "excluded_confirmed_near_duplicate_leakage.csv",
    "class_to_index": MANIFEST_ROOT / "metadata" / "class_to_index_main19.json",
    "index_to_class": MANIFEST_ROOT / "metadata" / "index_to_class_main19.json",
}

missing_paths = {name: str(path) for name, path in PATHS.items() if not path.exists()}

assert not missing_paths, f"Missing required files:\n{missing_paths}"

for name, path in PATHS.items():
    print(f"{name:34s} -> {path}")

main19_train                       -> /kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/split_outputs/split_outputs/manifests/main19/main19_train.csv
main19_calibration                 -> /kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/split_outputs/split_outputs/manifests/main19/main19_calibration.csv
main19_internal_test               -> /kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/split_outputs/split_outputs/manifests/main19/main19_internal_test.csv
review_progress                    -> /kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/split_outputs/split_outputs/tables/near_duplicate_manual_review_progress.csv
review_decisions                   -> /kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/split_outputs/split_outputs/tables/near_duplicate_review_decisions.csv
excluded_confirmed_near_dup        -> /kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/split_outputs/split_outputs/tables/excluded_confirmed_near_dupli

In [31]:
train_df = pd.read_csv(PATHS["main19_train"])
calib_df = pd.read_csv(PATHS["main19_calibration"])
test_df = pd.read_csv(PATHS["main19_internal_test"])

# Authoritative manual-review file.
review_progress_df = pd.read_csv(PATHS["review_progress"])

# Loaded only for transparency: this file may contain stale PENDING values.
review_decisions_df = pd.read_csv(PATHS["review_decisions"])

# Existing exclusion list, used as an independent verification source.
excluded_near_dup_df = pd.read_csv(PATHS["excluded_confirmed_near_dup"])

with open(PATHS["class_to_index"], "r", encoding="utf-8") as f:
    class_to_index = json.load(f)

with open(PATHS["index_to_class"], "r", encoding="utf-8") as f:
    index_to_class_raw = json.load(f)

index_to_class = {int(k): v for k, v in index_to_class_raw.items()}
class_names = [index_to_class[i] for i in range(NUM_CLASSES)]

print("Train:", train_df.shape)
print("Calibration:", calib_df.shape)
print("Original internal test:", test_df.shape)
print("Review progress:", review_progress_df.shape)
print("Existing excluded rows:", excluded_near_dup_df.shape)
print("Classes:", len(class_to_index))

assert len(class_to_index) == NUM_CLASSES
assert sorted(class_to_index.values()) == list(range(NUM_CLASSES))

Train: (18113, 21)
Calibration: (3197, 21)
Original internal test: (5322, 21)
Review progress: (136, 31)
Existing excluded rows: (2, 20)
Classes: 19


In [32]:
def normalize_relative_path(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().replace("\\", "/")

    for marker in ["/train/", "/val/", "train/", "val/"]:
        pos = value.find(marker)
        if pos >= 0:
            return value[pos + 1:] if marker.startswith("/") else value[pos:]

    return value.lstrip("/")


def harmonize_manifest(df, historical_split):
    required_cols = ["image_path", "relative_path", "label"]
    missing = [col for col in required_cols if col not in df.columns]
    assert not missing, f"{historical_split}: missing columns {missing}"

    out = df.copy().reset_index(drop=True)

    out["relative_path"] = out["relative_path"].apply(normalize_relative_path)
    out["image_id"] = out["relative_path"]
    out["class_name"] = out["label"].astype(str).str.strip()
    out["label_index"] = out["class_name"].map(class_to_index)

    assert out["relative_path"].notna().all()
    assert out["image_id"].notna().all()
    assert out["class_name"].notna().all()
    assert out["label_index"].notna().all(), (
        f"{historical_split}: unknown classes: "
        f"{out.loc[out['label_index'].isna(), 'class_name'].unique().tolist()}"
    )

    out["label_index"] = out["label_index"].astype(int)
    out["historical_split"] = historical_split

    keep_cols = [
        "image_id",
        "image_path",
        "relative_path",
        "class_name",
        "label_index",
        "historical_split",
        "sha256",
        "phash",
        "file_name",
        "crop",
        "disease",
    ]

    available_cols = [col for col in keep_cols if col in out.columns]
    out = out[available_cols].copy()

    return out


train_m = harmonize_manifest(train_df, "train")
calib_m = harmonize_manifest(calib_df, "calibration")
test_m = harmonize_manifest(test_df, "internal_test")

locked_main19 = pd.concat(
    [train_m, calib_m, test_m],
    ignore_index=True
)

assert len(locked_main19) == 26632
assert locked_main19["image_id"].is_unique
assert locked_main19["class_name"].nunique() == NUM_CLASSES

print("Locked Main19 shape:", locked_main19.shape)
print(locked_main19["historical_split"].value_counts())
display(locked_main19.head())

Locked Main19 shape: (26632, 11)
historical_split
train            18113
internal_test     5322
calibration       3197
Name: count, dtype: int64


,image_id,image_path,relative_path,class_name,label_index,historical_split,sha256,phash,file_name,crop,disease
0,train/Corn_(maize)___Common_rust_/RS_Rust 1915...,/kaggle/input/datasets/mohitsingh1804/plantvil...,train/Corn_(maize)___Common_rust_/RS_Rust 1915...,corn__common_rust,3,train,e12f62e6c710b002d2b4b9cbd39a30aeb3121da981efcb...,d0e21ec1b09fb596,RS_Rust 1915.JPG,corn,common_rust
1,train/Tomato___Bacterial_spot/e41ca477-b751-45...,/kaggle/input/datasets/mohitsingh1804/plantvil...,train/Tomato___Bacterial_spot/e41ca477-b751-45...,tomato__bacterial_spot,9,train,9d62d6860f47567f60ce3a636429c85c6ce0491c16a299...,ebc5c69f941a901b,e41ca477-b751-45f7-9a19-3442349e45ef___GCREC_B...,tomato,bacterial_spot
2,train/Tomato___Late_blight/370b9866-9a0d-4541-...,/kaggle/input/datasets/mohitsingh1804/plantvil...,train/Tomato___Late_blight/370b9866-9a0d-4541-...,tomato__late_blight,12,train,a79cf89e81955ad69c8089b51540c5d98601cebb068bcb...,ffd0e20f983cb222,370b9866-9a0d-4541-aaf1-cfefa1d2268c___RS_Late...,tomato,late_blight
3,train/Tomato___Bacterial_spot/509c3853-b405-44...,/kaggle/input/datasets/mohitsingh1804/plantvil...,train/Tomato___Bacterial_spot/509c3853-b405-44...,tomato__bacterial_spot,9,train,e0a3a12ae7d66317124213d1d0276050aeab1a96d1b32b...,eba5c45a9c399178,509c3853-b405-440f-bda3-59f1d7d238be___GCREC_B...,tomato,bacterial_spot
4,train/Tomato___Bacterial_spot/71eeeb09-ce59-4b...,/kaggle/input/datasets/mohitsingh1804/plantvil...,train/Tomato___Bacterial_spot/71eeeb09-ce59-4b...,tomato__bacterial_spot,9,train,a277cddb7216ca7b58c08cd35568ad70e640fb60382cd8...,eee1840e9678cda9,71eeeb09-ce59-4b58-80a4-a01c4348a218___GCREC_B...,tomato,bacterial_spot


In [33]:
assert set(locked_main19["class_name"]) == set(class_to_index)

mapping_check = locked_main19["class_name"].map(class_to_index)
assert (mapping_check == locked_main19["label_index"]).all()

original_split_counts = (
    locked_main19["historical_split"]
    .value_counts()
    .reindex(["train", "calibration", "internal_test"])
)

expected_original_counts = {
    "train": 18113,
    "calibration": 3197,
    "internal_test": 5322,
}

assert original_split_counts.to_dict() == expected_original_counts

class_distribution_original = (
    locked_main19
    .groupby(["historical_split", "class_name"])
    .size()
    .reset_index(name="n_images")
)

class_distribution_original.to_csv(
    OUTPUTS["reports"] / "original_main19_class_distribution.csv",
    index=False
)

print("Original split counts:")
display(original_split_counts)

print("Class-map verification passed.")

Original split counts:


historical_split
train            18113
calibration       3197
internal_test     5322
Name: count, dtype: int64

Class-map verification passed.


In [34]:
# The review_progress file is authoritative:
# it includes CONFIRMED_LEAKAGE / UNREVIEWED outcomes.
# The review_decisions file is not used to define exclusions because it
# currently contains PENDING placeholders.

review_source = review_progress_df.copy()

required_review_columns = [
    "candidate_id",
    "image_a_path",
    "image_b_path",
    "original_label",
    "review_decision",
    "review_reason",
    "exclude_image_path",
]

missing_review_cols = [
    col for col in required_review_columns
    if col not in review_source.columns
]

assert not missing_review_cols, f"Missing review columns: {missing_review_cols}"

review_source["review_decision"] = (
    review_source["review_decision"]
    .fillna("UNREVIEWED")
    .astype(str)
    .str.strip()
    .str.upper()
)

main19_labels = set(locked_main19["class_name"].unique())

main19_reviews = review_source.loc[
    review_source["original_label"].isin(main19_labels)
].copy()

completed_main19 = main19_reviews.loc[
    main19_reviews["review_decision"].isin(
        ["CONFIRMED_LEAKAGE", "NOT_LEAKAGE"]
    )
].copy()

confirmed_main19 = main19_reviews.loc[
    main19_reviews["review_decision"].eq("CONFIRMED_LEAKAGE")
].copy()

unresolved_main19 = main19_reviews.loc[
    main19_reviews["review_decision"].eq("UNREVIEWED")
].copy()

print("Main19 candidate pairs:", len(main19_reviews))
print("Completed manual decisions:", len(completed_main19))
print("Confirmed leakage pairs:", len(confirmed_main19))
print("Unresolved manual-review pairs:", len(unresolved_main19))
print()
print(main19_reviews["review_decision"].value_counts(dropna=False))

assert len(main19_reviews) == 11
assert len(completed_main19) == 2
assert len(confirmed_main19) == 2
assert len(unresolved_main19) == 9

display(
    main19_reviews[
        [
            "candidate_id",
            "original_label",
            "phash_hamming_distance",
            "image_a_path",
            "image_b_path",
            "review_decision",
            "review_reason",
            "exclude_image_path",
            "reviewer",
            "review_date",
        ]
    ].sort_values("candidate_id")
)

Main19 candidate pairs: 11
Completed manual decisions: 2
Confirmed leakage pairs: 2
Unresolved manual-review pairs: 9

review_decision
UNREVIEWED           9
CONFIRMED_LEAKAGE    2
Name: count, dtype: int64


,candidate_id,original_label,phash_hamming_distance,image_a_path,image_b_path,review_decision,review_reason,exclude_image_path,reviewer,review_date
0,1,corn__healthy,0,/kaggle/input/datasets/mohitsingh1804/plantvil...,/kaggle/input/datasets/mohitsingh1804/plantvil...,CONFIRMED_LEAKAGE,Manual visual review: same source leaf/image o...,/kaggle/input/datasets/mohitsingh1804/plantvil...,research_team,2026-08-12
1,2,potato__early_blight,2,/kaggle/input/datasets/mohitsingh1804/plantvil...,/kaggle/input/datasets/mohitsingh1804/plantvil...,CONFIRMED_LEAKAGE,Manual visual review: same source leaf/image o...,/kaggle/input/datasets/mohitsingh1804/plantvil...,research_team,2026-08-12
9,10,tomato__tomato_yellow_leaf_curl_virus,2,/kaggle/input/datasets/mohitsingh1804/plantvil...,/kaggle/input/datasets/mohitsingh1804/plantvil...,UNREVIEWED,Awaiting manual visual inspection,NaN,research_team,2026-08-12
10,11,bell_pepper__bacterial_spot,4,/kaggle/input/datasets/mohitsingh1804/plantvil...,/kaggle/input/datasets/mohitsingh1804/plantvil...,UNREVIEWED,Awaiting manual visual inspection,NaN,research_team,2026-08-12
11,12,bell_pepper__healthy,4,/kaggle/input/datasets/mohitsingh1804/plantvil...,/kaggle/input/datasets/mohitsingh1804/plantvil...,UNREVIEWED,Awaiting manual visual inspection,NaN,research_team,2026-08-12
12,13,bell_pepper__healthy,4,/kaggle/input/datasets/mohitsingh1804/plantvil...,/kaggle/input/datasets/mohitsingh1804/plantvil...,UNREVIEWED,Awaiting manual visual inspection,NaN,research_team,2026-08-12
13,14,corn__northern_leaf_blight,4,/kaggle/input/datasets/mohitsingh1804/plantvil...,/kaggle/input/datasets/mohitsingh1804/plantvil...,UNREVIEWED,Awaiting manual visual inspection,NaN,research_team,2026-08-12
132,133,tomato__bacterial_spot,4,/kaggle/input/datasets/mohitsingh1804/plantvil...,/kaggle/input/datasets/mohitsingh1804/plantvil...,UNREVIEWED,Awaiting manual visual inspection,NaN,research_team,2026-08-12
133,134,tomato__leaf_mold,4,/kaggle/input/datasets/mohitsingh1804/plantvil...,/kaggle/input/datasets/mohitsingh1804/plantvil...,UNREVIEWED,Awaiting manual visual inspection,NaN,research_team,2026-08-12
134,135,tomato__tomato_yellow_leaf_curl_virus,4,/kaggle/input/datasets/mohitsingh1804/plantvil...,/kaggle/input/datasets/mohitsingh1804/plantvil...,UNREVIEWED,Awaiting manual visual inspection,NaN,research_team,2026-08-12


In [35]:
# ============================================================
# Main19 duplicate audit and deterministic exclusion pipeline
#
# Exact SHA-256 policy:
# - Remove one row from every exact duplicate group that spans
#   multiple splits.
# - Prefer removing calibration, then train.
# - Never automatically remove an internal-test row.
#
# Manual pHash-review policy:
# - Remove a manually confirmed row only when its exclusion path
#   resolves uniquely to a current internal-test row.
# - If it cannot be reconciled, retain the data and report it.
#
# Within-split exact duplicates:
# - Retained in this run for split stability.
# - Reported separately for later optional deduplication.
# ============================================================

def clean_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    return value if value else np.nan


def clean_sha(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    return value if value else np.nan


def path_key(value):
    """
    Convert absolute or relative image paths to a split-agnostic key:
    ClassFolder/filename.ext

    Examples:
      /kaggle/.../train/Corn_(maize)___healthy/file.JPG
      val/Corn_(maize)___healthy/file.JPG
    become:
      Corn_(maize)___healthy/file.JPG
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip().replace("\\", "/")

    for marker in ["/train/", "/val/", "train/", "val/"]:
        position = value.find(marker)
        if position >= 0:
            value = value[position + len(marker):]
            break

    return value.lstrip("/")


def filename_key(value):
    if pd.isna(value):
        return np.nan
    return Path(str(value).replace("\\", "/")).name


def exact_duplicate_removal_priority(split_name):
    """
    Lower priority value means this split is preferred for removal.

    Calibration is removed first because it avoids changing the
    locked internal-test set. Train is removed next. Internal test
    is never automatically selected for removal.
    """
    priority = {
        "calibration": 0,
        "train": 1,
        "internal_test": 99,
    }
    return priority.get(split_name, 999)


# ------------------------------------------------------------
# Build audit-friendly keys for all current Main19 rows.
# ------------------------------------------------------------
audit_main19 = locked_main19.copy()

required_audit_columns = [
    "image_id",
    "relative_path",
    "image_path",
    "class_name",
    "historical_split",
    "sha256",
]

missing_audit_columns = [
    column
    for column in required_audit_columns
    if column not in audit_main19.columns
]

assert not missing_audit_columns, (
    f"Missing required Main19 audit columns: {missing_audit_columns}"
)

audit_main19["sha256_clean"] = audit_main19["sha256"].apply(clean_sha)
audit_main19["relative_path_key"] = audit_main19["relative_path"].apply(path_key)
audit_main19["image_path_key"] = audit_main19["image_path"].apply(path_key)
audit_main19["file_name_key"] = audit_main19["relative_path"].apply(filename_key)

assert audit_main19["image_id"].is_unique
assert audit_main19["sha256_clean"].notna().all()
assert audit_main19["relative_path_key"].notna().all()
assert audit_main19["image_path_key"].notna().all()

assert len(audit_main19) == 26632
assert audit_main19["historical_split"].value_counts().to_dict() == {
    "train": 18113,
    "calibration": 3197,
    "internal_test": 5322,
}

# ------------------------------------------------------------
# Exact SHA-256 duplicate audit.
# ------------------------------------------------------------
sha_counts = audit_main19["sha256_clean"].value_counts()

duplicate_shas = sha_counts.loc[sha_counts.gt(1)]

exact_duplicate_rows = (
    audit_main19.loc[
        audit_main19["sha256_clean"].isin(duplicate_shas.index),
        [
            "image_id",
            "relative_path",
            "class_name",
            "historical_split",
            "sha256_clean",
            "phash",
        ],
    ]
    .sort_values(["sha256_clean", "historical_split", "relative_path"])
    .reset_index(drop=True)
)

exact_duplicate_summary = (
    audit_main19.groupby("sha256_clean", as_index=False)
    .agg(
        n_rows=("image_id", "size"),
        n_splits=("historical_split", "nunique"),
        split_names=("historical_split", lambda x: "|".join(sorted(set(x)))),
        n_classes=("class_name", "nunique"),
        class_names=("class_name", lambda x: "|".join(sorted(set(x)))),
    )
    .query("n_rows > 1")
    .sort_values(["n_splits", "n_rows", "sha256_clean"], ascending=[False, False, True])
    .reset_index(drop=True)
)

cross_split_exact_summary = exact_duplicate_summary.loc[
    exact_duplicate_summary["n_splits"] > 1
].copy()

cross_split_exact_shas = cross_split_exact_summary["sha256_clean"].tolist()

cross_split_exact_rows = (
    audit_main19.loc[
        audit_main19["sha256_clean"].isin(cross_split_exact_shas),
        [
            "image_id",
            "relative_path",
            "class_name",
            "historical_split",
            "sha256_clean",
            "phash",
        ],
    ]
    .sort_values(["sha256_clean", "historical_split", "relative_path"])
    .reset_index(drop=True)
)

within_split_exact_summary = exact_duplicate_summary.loc[
    exact_duplicate_summary["n_splits"] == 1
].copy()

print("Unique SHA-256 values:", audit_main19["sha256_clean"].nunique())
print("Rows in Main19:", len(audit_main19))
print("SHA-256 values appearing more than once:", len(duplicate_shas))
print("Rows participating in exact SHA-256 duplicates:", len(exact_duplicate_rows))
print("Exact SHA-256 duplicate groups spanning multiple splits:", len(cross_split_exact_shas))
print("Exact SHA-256 duplicate groups within one split only:", len(within_split_exact_summary))

print("\nAll exact SHA-256 duplicate rows:")
display(exact_duplicate_rows)

print("\nCross-split exact SHA-256 duplicate rows:")
display(cross_split_exact_rows)

print("\nExact SHA-256 duplicate-group summary:")
display(exact_duplicate_summary)

# ------------------------------------------------------------
# Deterministically remove one row per cross-split exact-SHA group.
#
# Guardrail: if an exact duplicate group includes internal_test,
# this code refuses automatic removal, because the internal-test
# split must remain locked. Such cases must be handled manually.
# ------------------------------------------------------------
cross_split_groups_with_test = cross_split_exact_summary.loc[
    cross_split_exact_summary["split_names"].str.contains(
        "internal_test",
        regex=False,
        na=False,
    )
].copy()

if len(cross_split_groups_with_test) > 0:
    print(
        "\nWARNING: Exact SHA-256 duplicate groups involving internal_test "
        "were found. They will NOT be removed automatically."
    )
    display(cross_split_groups_with_test)

cross_split_exact_rows_safe = cross_split_exact_rows.loc[
    ~cross_split_exact_rows["sha256_clean"].isin(
        set(cross_split_groups_with_test["sha256_clean"])
    )
].copy()

exact_cross_split_rows_to_remove = (
    cross_split_exact_rows_safe.assign(
        removal_priority=lambda df: df["historical_split"].map(
            exact_duplicate_removal_priority
        )
    )
    .sort_values(
        ["sha256_clean", "removal_priority", "relative_path"]
    )
    .groupby("sha256_clean", group_keys=False)
    .head(1)
    .copy()
)

exact_cross_split_rows_to_remove["exclusion_reason"] = (
    "EXACT_SHA256_CROSS_SPLIT_DUPLICATE"
)
exact_cross_split_rows_to_remove["exclusion_source"] = (
    "automatic_sha256_policy"
)

assert exact_cross_split_rows_to_remove["historical_split"].isin(
    ["train", "calibration"]
).all(), (
    "Automatic exact-SHA removal attempted to remove an internal-test row."
)

assert exact_cross_split_rows_to_remove["sha256_clean"].is_unique

print("\nExact SHA-256 rows selected for removal:")
display(
    exact_cross_split_rows_to_remove[
        [
            "image_id",
            "relative_path",
            "class_name",
            "historical_split",
            "sha256_clean",
            "exclusion_reason",
            "exclusion_source",
        ]
    ]
)

print(
    "Rows selected for cross-split exact-SHA removal:",
    len(exact_cross_split_rows_to_remove),
)

# ------------------------------------------------------------
# Persist exact-duplicate audit outputs before manual review.
# ------------------------------------------------------------
exact_duplicate_rows.to_csv(
    OUTPUTS["audit"] / "main19_exact_sha256_duplicate_rows.csv",
    index=False,
)

exact_duplicate_summary.to_csv(
    OUTPUTS["audit"] / "main19_exact_sha256_duplicate_summary.csv",
    index=False,
)

cross_split_exact_rows.to_csv(
    OUTPUTS["audit"] / "main19_cross_split_exact_sha256_duplicate_rows.csv",
    index=False,
)

exact_cross_split_rows_to_remove.to_csv(
    OUTPUTS["audit"] / "main19_excluded_exact_sha256_cross_split_rows.csv",
    index=False,
)

within_split_exact_summary.to_csv(
    OUTPUTS["audit"] / "main19_within_split_exact_sha256_duplicate_summary.csv",
    index=False,
)

# ------------------------------------------------------------
# Confirmed manual-review rows are authoritative.
# ------------------------------------------------------------
confirmed_exclusions = confirmed_main19.copy()

assert len(confirmed_exclusions) == 2, (
    f"Expected exactly 2 CONFIRMED_LEAKAGE rows; found "
    f"{len(confirmed_exclusions)}."
)

assert confirmed_exclusions["exclude_image_path"].notna().all(), (
    "Every confirmed leakage record must have an exclude_image_path."
)

confirmed_exclusions["exclude_path_key"] = (
    confirmed_exclusions["exclude_image_path"].apply(path_key)
)

confirmed_exclusions["exclude_file_name_key"] = (
    confirmed_exclusions["exclude_image_path"].apply(filename_key)
)

print("\nAuthoritative confirmed manual-review leakage records:")
display(
    confirmed_exclusions[
        [
            "candidate_id",
            "original_label",
            "image_a_path",
            "image_b_path",
            "exclude_image_path",
            "exclude_path_key",
            "exclude_file_name_key",
            "review_decision",
            "review_reason",
        ]
    ]
)

# ------------------------------------------------------------
# Resolve each manually confirmed exclusion.
#
# Resolution priority:
# 1. Exact normalised relative_path
# 2. Exact normalised image_path
# 3. Unique filename fallback
#
# Rows are automatically removed only when each review record
# resolves uniquely to an internal-test image.
# ------------------------------------------------------------
resolution_rows = []

for _, review_row in confirmed_exclusions.iterrows():
    by_relative_path = audit_main19.loc[
        audit_main19["relative_path_key"].eq(review_row["exclude_path_key"])
    ].copy()

    by_image_path = audit_main19.loc[
        audit_main19["image_path_key"].eq(review_row["exclude_path_key"])
    ].copy()

    by_filename = audit_main19.loc[
        audit_main19["file_name_key"].eq(review_row["exclude_file_name_key"])
    ].copy()

    resolved = pd.DataFrame()
    resolution_method = "unresolved"

    if len(by_relative_path) == 1:
        resolved = by_relative_path.copy()
        resolution_method = "relative_path_key"

    elif len(by_relative_path) > 1:
        resolved = by_relative_path.copy()
        resolution_method = "ambiguous_relative_path_key"

    elif len(by_image_path) == 1:
        resolved = by_image_path.copy()
        resolution_method = "image_path_key"

    elif len(by_image_path) > 1:
        resolved = by_image_path.copy()
        resolution_method = "ambiguous_image_path_key"

    elif len(by_filename) == 1:
        resolved = by_filename.copy()
        resolution_method = "unique_filename_fallback"

    elif len(by_filename) > 1:
        resolved = by_filename.copy()
        resolution_method = "ambiguous_filename_fallback"

    if len(resolved) == 0:
        resolution_rows.append(
            {
                "candidate_id": review_row["candidate_id"],
                "original_label": review_row["original_label"],
                "exclude_image_path": review_row["exclude_image_path"],
                "exclude_path_key": review_row["exclude_path_key"],
                "resolution_method": resolution_method,
                "resolved_image_id": np.nan,
                "resolved_relative_path": np.nan,
                "resolved_historical_split": np.nan,
                "resolved_sha256_clean": np.nan,
                "n_relative_path_matches": len(by_relative_path),
                "n_image_path_matches": len(by_image_path),
                "n_filename_matches": len(by_filename),
            }
        )
    else:
        for _, resolved_row in resolved.iterrows():
            resolution_rows.append(
                {
                    "candidate_id": review_row["candidate_id"],
                    "original_label": review_row["original_label"],
                    "exclude_image_path": review_row["exclude_image_path"],
                    "exclude_path_key": review_row["exclude_path_key"],
                    "resolution_method": resolution_method,
                    "resolved_image_id": resolved_row["image_id"],
                    "resolved_relative_path": resolved_row["relative_path"],
                    "resolved_historical_split": resolved_row["historical_split"],
                    "resolved_sha256_clean": resolved_row["sha256_clean"],
                    "n_relative_path_matches": len(by_relative_path),
                    "n_image_path_matches": len(by_image_path),
                    "n_filename_matches": len(by_filename),
                }
            )

resolution_df = pd.DataFrame(resolution_rows)

print("\nManual-review resolution report:")
display(resolution_df)

resolution_df.to_csv(
    OUTPUTS["audit"] / "main19_confirmed_exclusion_resolution_report.csv",
    index=False,
)

allowed_manual_resolution_methods = {
    "relative_path_key",
    "image_path_key",
    "unique_filename_fallback",
}

resolved_test_exclusions = resolution_df.loc[
    resolution_df["resolution_method"].isin(allowed_manual_resolution_methods)
    & resolution_df["resolved_historical_split"].eq("internal_test")
    & resolution_df["resolved_image_id"].notna()
].copy()

resolved_test_exclusions = resolved_test_exclusions.drop_duplicates(
    subset=["resolved_image_id"]
).reset_index(drop=True)

n_confirmed_records = len(confirmed_exclusions)
n_resolved_test_targets = len(resolved_test_exclusions)

print("\nUnique manual exclusions resolved to current internal-test rows:")
display(
    resolved_test_exclusions[
        [
            "candidate_id",
            "original_label",
            "resolution_method",
            "resolved_image_id",
            "resolved_relative_path",
            "resolved_historical_split",
            "resolved_sha256_clean",
        ]
    ]
)

# ------------------------------------------------------------
# Apply manual-review exclusions only when BOTH confirmed records
# resolve uniquely to distinct current internal-test rows.
# ------------------------------------------------------------
if n_resolved_test_targets == n_confirmed_records:
    manual_excluded_ids = set(resolved_test_exclusions["resolved_image_id"])

    manual_excluded_rows = audit_main19.loc[
        audit_main19["image_id"].isin(manual_excluded_ids)
    ].copy()

    assert len(manual_excluded_rows) == n_confirmed_records
    assert manual_excluded_rows["historical_split"].eq("internal_test").all()

    manual_excluded_rows["exclusion_reason"] = (
        "CONFIRMED_MANUAL_NEAR_DUPLICATE_LEAKAGE"
    )
    manual_excluded_rows["exclusion_source"] = (
        "resolved_manual_review_record"
    )

    manual_review_action = (
        "removed_resolved_manual_review_confirmed_internal_test_rows"
    )

else:
    manual_excluded_rows = audit_main19.iloc[0:0].copy()
    manual_excluded_rows["exclusion_reason"] = pd.Series(dtype="object")
    manual_excluded_rows["exclusion_source"] = pd.Series(dtype="object")

    manual_review_action = (
        "manual_review_paths_not_uniquely_reconciled_no_manual_rows_removed"
    )

    print(
        "\nWARNING: The manually confirmed exclusions were not all resolved "
        "uniquely to current internal-test rows. No manual-review rows were "
        "removed automatically. Review the resolution report."
    )

# ------------------------------------------------------------
# Merge exact-SHA and manual-review exclusions.
# ------------------------------------------------------------
manual_excluded_columns = [
    "image_id",
    "relative_path",
    "class_name",
    "historical_split",
    "sha256_clean",
    "exclusion_reason",
    "exclusion_source",
]

manual_excluded_for_merge = manual_excluded_rows.reindex(
    columns=manual_excluded_columns
).copy()

exact_excluded_for_merge = exact_cross_split_rows_to_remove.reindex(
    columns=manual_excluded_columns
).copy()

all_excluded_rows = (
    pd.concat(
        [exact_excluded_for_merge, manual_excluded_for_merge],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(subset=["image_id"])
    .sort_values(["historical_split", "class_name", "relative_path"])
    .reset_index(drop=True)
)

excluded_image_ids = set(all_excluded_rows["image_id"])

assert len(excluded_image_ids) == len(all_excluded_rows)
assert not all_excluded_rows["image_id"].duplicated().any()

print("\nAll exclusions applied in this notebook:")
display(all_excluded_rows)

# ------------------------------------------------------------
# Create audited split manifests.
# ------------------------------------------------------------
audited_main19 = audit_main19.loc[
    ~audit_main19["image_id"].isin(excluded_image_ids)
].copy()

audited_train = audited_main19.loc[
    audited_main19["historical_split"].eq("train")
].copy()

audited_calib = audited_main19.loc[
    audited_main19["historical_split"].eq("calibration")
].copy()

audited_test = audited_main19.loc[
    audited_main19["historical_split"].eq("internal_test")
].copy()

audited_split_counts = {
    "train": len(audited_train),
    "calibration": len(audited_calib),
    "internal_test": len(audited_test),
}

assert audited_main19["image_id"].is_unique
assert audited_main19["class_name"].nunique() == NUM_CLASSES

# Exact-SHA policy must never alter the internal-test count.
assert len(audited_test) == 5322

# Exact SHA-256 audit result for the current frozen Main19 manifests.
# Three duplicate groups span train and calibration, and the calibration-side
# member of each group is removed to preserve the larger training partition.
EXPECTED_EXACT_CROSS_SPLIT_GROUPS = 3
EXPECTED_EXACT_CROSS_SPLIT_ROWS_TO_REMOVE = 3

assert len(cross_split_exact_shas) == EXPECTED_EXACT_CROSS_SPLIT_GROUPS, (
    f"Expected {EXPECTED_EXACT_CROSS_SPLIT_GROUPS} cross-split exact-SHA-256 "
    f"duplicate groups, found {len(cross_split_exact_shas)}. "
    "Inspect the duplicate audit because the input manifests may have changed."
)

assert len(exact_cross_split_rows_to_remove) == EXPECTED_EXACT_CROSS_SPLIT_ROWS_TO_REMOVE, (
    f"Expected {EXPECTED_EXACT_CROSS_SPLIT_ROWS_TO_REMOVE} removable "
    f"cross-split exact-SHA-256 rows, found "
    f"{len(exact_cross_split_rows_to_remove)}. "
    "Inspect the duplicate audit because the input manifests may have changed."
)

assert set(exact_cross_split_rows_to_remove["historical_split"]) == {"calibration"}, (
    "The exact-SHA removal policy expects every removed cross-split duplicate "
    "to come from the calibration partition."
)

assert exact_cross_split_rows_to_remove["sha256_clean"].nunique() == 3, (
    "Expected one selected removal from each cross-split exact-SHA-256 group."
)

print(
    f"Exact-SHA audit passed: "
    f"{len(cross_split_exact_shas)} cross-split groups, "
    f"{len(exact_cross_split_rows_to_remove)} calibration rows removed."
)

# Final audited partition counts.
# Only three calibration rows were removed automatically because they were
# exact SHA-256 duplicates of training images. Manual-review exclusions were
# not applied because neither record could be uniquely resolved to a current row.
EXPECTED_AUDITED_TRAIN = 18113
EXPECTED_AUDITED_CALIBRATION = 3194
EXPECTED_AUDITED_INTERNAL_TEST = 5322
EXPECTED_AUDITED_TOTAL = (
    EXPECTED_AUDITED_TRAIN
    + EXPECTED_AUDITED_CALIBRATION
    + EXPECTED_AUDITED_INTERNAL_TEST
)

assert len(audited_train) == EXPECTED_AUDITED_TRAIN, (
    f"Expected {EXPECTED_AUDITED_TRAIN} audited training rows; "
    f"found {len(audited_train)}."
)

assert len(audited_calib) == EXPECTED_AUDITED_CALIBRATION, (
    f"Expected {EXPECTED_AUDITED_CALIBRATION} audited calibration rows; "
    f"found {len(audited_calib)}."
)

assert len(audited_test) == EXPECTED_AUDITED_INTERNAL_TEST, (
    f"Expected {EXPECTED_AUDITED_INTERNAL_TEST} audited internal-test rows; "
    f"found {len(audited_test)}."
)

assert len(audited_main19) == EXPECTED_AUDITED_TOTAL, (
    f"Expected {EXPECTED_AUDITED_TOTAL} audited Main19 rows; "
    f"found {len(audited_main19)}."
)

assert len(all_excluded_rows) == 3, (
    f"Expected exactly 3 applied exclusions; found {len(all_excluded_rows)}."
)

assert len(manual_excluded_rows) == 0, (
    "Manual-review rows should not be removed while their paths remain unresolved."
)

print("Final audited Main19 counts:")
print(f"  Train:         {len(audited_train):,}")
print(f"  Calibration:   {len(audited_calib):,}")
print(f"  Internal test: {len(audited_test):,}")
print(f"  Total:         {len(audited_main19):,}")

if n_resolved_test_targets == n_confirmed_records:
    assert len(audited_test) == 5320
    assert len(audited_main19) == 26626
else:
    assert len(audited_test) == 5322
    assert len(audited_main19) == 26629

if (
    len(exact_cross_split_rows_to_remove) > 0
    and len(manual_excluded_rows) > 0
):
    audit_action = (
        "removed_exact_sha256_cross_split_duplicates_"
        "and_resolved_manual_review_exclusions"
    )
elif len(exact_cross_split_rows_to_remove) > 0:
    audit_action = (
        "removed_exact_sha256_cross_split_duplicates_"
        "manual_review_paths_not_uniquely_reconciled"
    )
elif len(manual_excluded_rows) > 0:
    audit_action = "removed_resolved_manual_review_exclusions_only"
else:
    audit_action = "no_rows_removed"

print("\nAudit action:", audit_action)
print("Manual review action:", manual_review_action)
print("Original Main19 rows:", len(audit_main19))
print("Exact SHA cross-split rows removed:", len(exact_cross_split_rows_to_remove))
print("Manual-review rows removed:", len(manual_excluded_rows))
print("Total unique rows removed:", len(all_excluded_rows))
print("Audited Main19 rows:", len(audited_main19))
print("Audited split counts:", audited_split_counts)

# ------------------------------------------------------------
# Persist audit and cleaned manifest outputs.
# ------------------------------------------------------------
audited_train.to_csv(
    OUTPUTS["manifests"] / "main19_audited_train.csv",
    index=False,
)

audited_calib.to_csv(
    OUTPUTS["manifests"] / "main19_audited_calibration.csv",
    index=False,
)

audited_test.to_csv(
    OUTPUTS["manifests"] / "main19_audited_internal_test.csv",
    index=False,
)

audited_main19.to_csv(
    OUTPUTS["manifests"] / "main19_audited_all_retained.csv",
    index=False,
)

manual_excluded_rows.to_csv(
    OUTPUTS["audit"] / "main19_excluded_confirmed_near_duplicate_images.csv",
    index=False,
)

resolved_test_exclusions.to_csv(
    OUTPUTS["audit"] / "main19_resolved_confirmed_exclusions.csv",
    index=False,
)

all_excluded_rows.to_csv(
    OUTPUTS["audit"] / "main19_all_excluded_rows.csv",
    index=False,
)

audit_summary = pd.DataFrame(
    {
        "metric": [
            "original_locked_main19_images",
            "unique_sha256_values",
            "sha256_values_appearing_more_than_once",
            "rows_participating_in_exact_sha256_duplicates",
            "within_split_exact_sha256_duplicate_groups",
            "cross_split_exact_sha256_duplicate_groups",
            "cross_split_exact_sha256_rows_selected_for_removal",
            "cross_split_exact_sha256_groups_involving_internal_test",
            "main19_candidate_pairs",
            "confirmed_manual_review_pairs",
            "unresolved_manual_review_pairs",
            "confirmed_exclusions_resolved_to_internal_test",
            "manual_review_rows_removed",
            "total_rows_removed_in_this_notebook",
            "audit_action",
            "manual_review_action",
            "audited_main19_images",
            "audited_train_images",
            "audited_calibration_images",
            "audited_internal_test_images",
        ],
        "value": [
            len(audit_main19),
            audit_main19["sha256_clean"].nunique(),
            len(duplicate_shas),
            len(exact_duplicate_rows),
            len(within_split_exact_summary),
            len(cross_split_exact_shas),
            len(exact_cross_split_rows_to_remove),
            len(cross_split_groups_with_test),
            len(main19_reviews),
            len(confirmed_main19),
            len(unresolved_main19),
            n_resolved_test_targets,
            len(manual_excluded_rows),
            len(all_excluded_rows),
            audit_action,
            manual_review_action,
            len(audited_main19),
            len(audited_train),
            len(audited_calib),
            len(audited_test),
        ],
    }
)

audit_summary.to_csv(
    OUTPUTS["audit"] / "main19_duplicate_audit_summary.csv",
    index=False,
)

display(audit_summary)

# ------------------------------------------------------------
# Final consistency checks for saved artefacts.
# ------------------------------------------------------------
assert (
    OUTPUTS["audit"] / "main19_exact_sha256_duplicate_rows.csv"
).exists()

assert (
    OUTPUTS["audit"] / "main19_cross_split_exact_sha256_duplicate_rows.csv"
).exists()

assert (
    OUTPUTS["audit"] / "main19_excluded_exact_sha256_cross_split_rows.csv"
).exists()

assert (
    OUTPUTS["audit"] / "main19_all_excluded_rows.csv"
).exists()

assert (
    OUTPUTS["manifests"] / "main19_audited_train.csv"
).exists()

assert (
    OUTPUTS["manifests"] / "main19_audited_calibration.csv"
).exists()

assert (
    OUTPUTS["manifests"] / "main19_audited_internal_test.csv"
).exists()

assert len(
    pd.read_csv(
        OUTPUTS["manifests"] / "main19_audited_calibration.csv"
    )
) == 3193

print("\nDuplicate audit and cleaned-manifest generation completed successfully.")

Unique SHA-256 values: 26623
Rows in Main19: 26632
SHA-256 values appearing more than once: 9
Rows participating in exact SHA-256 duplicates: 18
Exact SHA-256 duplicate groups spanning multiple splits: 3
Exact SHA-256 duplicate groups within one split only: 6

All exact SHA-256 duplicate rows:


,image_id,relative_path,class_name,historical_split,sha256_clean,phash
0,train/Tomato___Late_blight/5f21282c-e2ef-4c4a-...,train/Tomato___Late_blight/5f21282c-e2ef-4c4a-...,tomato__late_blight,train,2aeb7479e975df37dfb54aa7b403f57f2ad90b018c3780...,ad8e53b14cc65999
1,train/Tomato___Late_blight/e5d707cd-077c-43af-...,train/Tomato___Late_blight/e5d707cd-077c-43af-...,tomato__late_blight,train,2aeb7479e975df37dfb54aa7b403f57f2ad90b018c3780...,ad8e53b14cc65999
2,train/Tomato___healthy/9662364c-aaba-45e3-b907...,train/Tomato___healthy/9662364c-aaba-45e3-b907...,tomato__healthy,calibration,32e9ffabaa054c19ae8df95e94fbd7a25105defdea1214...,f9b2e0df8cc8246c
3,train/Tomato___healthy/d2ce7896-6fa4-45e6-96c5...,train/Tomato___healthy/d2ce7896-6fa4-45e6-96c5...,tomato__healthy,calibration,32e9ffabaa054c19ae8df95e94fbd7a25105defdea1214...,f9b2e0df8cc8246c
4,train/Tomato___Late_blight/48c55974-9fe9-4f4b-...,train/Tomato___Late_blight/48c55974-9fe9-4f4b-...,tomato__late_blight,train,375eee7b5da77c7afdd73d089574bee3148d04cbe266ee...,9931664c736639e6
5,train/Tomato___Late_blight/bd4f09bd-ee85-4ab1-...,train/Tomato___Late_blight/bd4f09bd-ee85-4ab1-...,tomato__late_blight,train,375eee7b5da77c7afdd73d089574bee3148d04cbe266ee...,9931664c736639e6
6,train/Tomato___Late_blight/861d8b00-57a4-46b0-...,train/Tomato___Late_blight/861d8b00-57a4-46b0-...,tomato__late_blight,calibration,3a60a7b6f3a287981d21dca37f27e5480cc46d1144688d...,c8e7653c32c1619f
7,train/Tomato___Late_blight/d82b634e-9a19-497b-...,train/Tomato___Late_blight/d82b634e-9a19-497b-...,tomato__late_blight,train,3a60a7b6f3a287981d21dca37f27e5480cc46d1144688d...,c8e7653c32c1619f
8,train/Tomato___Late_blight/3fae9c64-18f0-4a67-...,train/Tomato___Late_blight/3fae9c64-18f0-4a67-...,tomato__late_blight,calibration,4913a3835e692530e507d542caccacc0d0a7b2fd708782...,9cc7653836c6693c
9,train/Tomato___Late_blight/5de6da85-f8c4-48c4-...,train/Tomato___Late_blight/5de6da85-f8c4-48c4-...,tomato__late_blight,train,4913a3835e692530e507d542caccacc0d0a7b2fd708782...,9cc7653836c6693c



Cross-split exact SHA-256 duplicate rows:


,image_id,relative_path,class_name,historical_split,sha256_clean,phash
0,train/Tomato___Late_blight/861d8b00-57a4-46b0-...,train/Tomato___Late_blight/861d8b00-57a4-46b0-...,tomato__late_blight,calibration,3a60a7b6f3a287981d21dca37f27e5480cc46d1144688d...,c8e7653c32c1619f
1,train/Tomato___Late_blight/d82b634e-9a19-497b-...,train/Tomato___Late_blight/d82b634e-9a19-497b-...,tomato__late_blight,train,3a60a7b6f3a287981d21dca37f27e5480cc46d1144688d...,c8e7653c32c1619f
2,train/Tomato___Late_blight/3fae9c64-18f0-4a67-...,train/Tomato___Late_blight/3fae9c64-18f0-4a67-...,tomato__late_blight,calibration,4913a3835e692530e507d542caccacc0d0a7b2fd708782...,9cc7653836c6693c
3,train/Tomato___Late_blight/5de6da85-f8c4-48c4-...,train/Tomato___Late_blight/5de6da85-f8c4-48c4-...,tomato__late_blight,train,4913a3835e692530e507d542caccacc0d0a7b2fd708782...,9cc7653836c6693c
4,train/Tomato___healthy/1af0bfe1-4bcf-4b8b-be66...,train/Tomato___healthy/1af0bfe1-4bcf-4b8b-be66...,tomato__healthy,calibration,61f6fa6195ef5c59c4918a292c641d4cd3d2d31ff051dc...,8cf171cd06c6732e
5,train/Tomato___healthy/cfd491d6-4af5-4728-8f0e...,train/Tomato___healthy/cfd491d6-4af5-4728-8f0e...,tomato__healthy,train,61f6fa6195ef5c59c4918a292c641d4cd3d2d31ff051dc...,8cf171cd06c6732e



Exact SHA-256 duplicate-group summary:


...



All exclusions applied in this notebook:


,image_id,relative_path,class_name,historical_split,sha256_clean,exclusion_reason,exclusion_source
0,train/Tomato___healthy/1af0bfe1-4bcf-4b8b-be66...,train/Tomato___healthy/1af0bfe1-4bcf-4b8b-be66...,tomato__healthy,calibration,61f6fa6195ef5c59c4918a292c641d4cd3d2d31ff051dc...,EXACT_SHA256_CROSS_SPLIT_DUPLICATE,automatic_sha256_policy
1,train/Tomato___Late_blight/3fae9c64-18f0-4a67-...,train/Tomato___Late_blight/3fae9c64-18f0-4a67-...,tomato__late_blight,calibration,4913a3835e692530e507d542caccacc0d0a7b2fd708782...,EXACT_SHA256_CROSS_SPLIT_DUPLICATE,automatic_sha256_policy
2,train/Tomato___Late_blight/861d8b00-57a4-46b0-...,train/Tomato___Late_blight/861d8b00-57a4-46b0-...,tomato__late_blight,calibration,3a60a7b6f3a287981d21dca37f27e5480cc46d1144688d...,EXACT_SHA256_CROSS_SPLIT_DUPLICATE,automatic_sha256_policy


Exact-SHA audit passed: 3 cross-split groups, 3 calibration rows removed.
Final audited Main19 counts:
  Train:         18,113
  Calibration:   3,194
  Internal test: 5,322
  Total:         26,629

Audit action: removed_exact_sha256_cross_split_duplicates_manual_review_paths_not_uniquely_reconciled
Manual review action: manual_review_paths_not_uniquely_reconciled_no_manual_rows_removed
Original Main19 rows: 26632
Exact SHA cross-split rows removed: 3
Manual-review rows removed: 0
Total unique rows removed: 3
Audited Main19 rows: 26629
Audited split counts: {'train': 18113, 'calibration': 3194, 'internal_test': 5322}


,metric,value
0,original_locked_main19_images,26632
1,unique_sha256_values,26623
2,sha256_values_appearing_more_than_once,9
3,rows_participating_in_exact_sha256_duplicates,18
4,within_split_exact_sha256_duplicate_groups,6
5,cross_split_exact_sha256_duplicate_groups,3
6,cross_split_exact_sha256_rows_selected_for_rem...,3
7,cross_split_exact_sha256_groups_involving_inte...,0
8,main19_candidate_pairs,11
9,confirmed_manual_review_pairs,2


AssertionError: 

In [ ]:
main19_reviews.to_csv(
    OUTPUTS["audit"] / "main19_candidate_pairs_with_review_status.csv",
    index=False
)

confirmed_main19.to_csv(
    OUTPUTS["audit"] / "main19_confirmed_leakage_pairs.csv",
    index=False
)

unresolved_main19.to_csv(
    OUTPUTS["audit"] / "main19_unresolved_near_duplicate_pairs.csv",
    index=False
)

excluded_confirmed_rows.to_csv(
    OUTPUTS["audit"] / "main19_excluded_confirmed_near_duplicate_images.csv",
    index=False
)

audited_train.to_csv(
    OUTPUTS["manifests"] / "main19_audited_train.csv",
    index=False
)

audited_calib.to_csv(
    OUTPUTS["manifests"] / "main19_audited_calibration.csv",
    index=False
)

audited_test.to_csv(
    OUTPUTS["manifests"] / "main19_audited_internal_test.csv",
    index=False
)

audited_main19.to_csv(
    OUTPUTS["manifests"] / "main19_audited_all_retained.csv",
    index=False
)

audit_summary = pd.DataFrame(
    {
        "metric": [
            "original_locked_main19_images",
            "main19_candidate_pairs",
            "completed_manual_decisions",
            "confirmed_leakage_pairs",
            "unresolved_manual_review_pairs",
            "excluded_confirmed_leakage_images",
            "audited_main19_images",
            "audited_train_images",
            "audited_calibration_images",
            "audited_internal_test_images",
        ],
        "value": [
            len(locked_main19),
            len(main19_reviews),
            len(completed_main19),
            len(confirmed_main19),
            len(unresolved_main19),
            len(excluded_confirmed_rows),
            len(audited_main19),
            len(audited_train),
            len(audited_calib),
            len(audited_test),
        ],
    }
)

audit_summary.to_csv(
    OUTPUTS["audit"] / "main19_duplicate_audit_summary.csv",
    index=False
)

display(audit_summary)

assert (OUTPUTS["audit"] / "main19_unresolved_near_duplicate_pairs.csv").exists()
assert len(pd.read_csv(
    OUTPUTS["audit"] / "main19_unresolved_near_duplicate_pairs.csv"
)) == 9

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

audit_labels = [
    "Main19\ncandidates",
    "Completed\nreviews",
    "Confirmed\nleakage",
    "Unresolved\npairs",
]
audit_values = [
    len(main19_reviews),
    len(completed_main19),
    len(confirmed_main19),
    len(unresolved_main19),
]

colors = ["#4C78A8", "#72B7B2", "#E45756", "#F2CF5B"]

axes[0].bar(audit_labels, audit_values, color=colors)
axes[0].set_title("Main19 Near-Duplicate Audit")
axes[0].set_ylabel("Number of candidate pairs")
for i, value in enumerate(audit_values):
    axes[0].text(i, value + 0.2, str(value), ha="center", fontweight="bold")

split_labels = ["Train", "Calibration", "Internal test"]
before = [18113, 3197, 5322]
after = [18113, 3197, 5320]
x = np.arange(len(split_labels))
width = 0.36

axes[1].bar(x - width / 2, before, width, label="Original", color="#9ECAE1")
axes[1].bar(x + width / 2, after, width, label="Audited", color="#3182BD")
axes[1].set_xticks(x)
axes[1].set_xticklabels(split_labels)
axes[1].set_ylabel("Images")
axes[1].set_title("Main19 Split Counts Before and After Audit")
axes[1].legend()

for idx, value in enumerate(before):
    axes[1].text(idx - width / 2, value + 120, str(value), ha="center", fontsize=9)

for idx, value in enumerate(after):
    axes[1].text(idx + width / 2, value + 120, str(value), ha="center", fontsize=9)

plt.tight_layout()
fig.savefig(
    OUTPUTS["figures"] / "main19_duplicate_audit_and_split_counts.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
class_counts = (
    audited_main19
    .groupby(["historical_split", "class_name"])
    .size()
    .unstack(fill_value=0)
    .T
)

class_counts["total"] = class_counts.sum(axis=1)
class_counts = class_counts.sort_values("total", ascending=False)

class_counts.to_csv(
    OUTPUTS["reports"] / "audited_main19_class_distribution.csv"
)

assert (class_counts[["train", "calibration", "internal_test"]] >= 1).all().all()

display(class_counts)

fig, ax = plt.subplots(figsize=(14, 8))
class_counts[["train", "calibration", "internal_test"]].plot(
    kind="barh",
    stacked=True,
    ax=ax,
    color=["#4C78A8", "#72B7B2", "#E45756"],
)
ax.set_title("Audited Main19 Class Distribution by Partition")
ax.set_xlabel("Image count")
ax.set_ylabel("Class")
ax.legend(title="Partition")
plt.tight_layout()

fig.savefig(
    OUTPUTS["figures"] / "audited_main19_class_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
def check_image_manifest(df, split_name, sample_size=500):
    sample = df.sample(
        n=min(sample_size, len(df)),
        random_state=SEED
    ).copy()

    exists = sample["image_path"].map(lambda p: Path(p).exists())

    report = {
        "split": split_name,
        "n_images": len(df),
        "sample_checked": len(sample),
        "missing_paths_in_sample": int((~exists).sum()),
    }

    if (~exists).any():
        display(sample.loc[~exists, ["image_path", "relative_path", "class_name"]])

    return report


path_checks = pd.DataFrame(
    [
        check_image_manifest(audited_train, "train"),
        check_image_manifest(audited_calib, "calibration"),
        check_image_manifest(audited_test, "internal_test"),
    ]
)

display(path_checks)

assert path_checks["missing_paths_in_sample"].sum() == 0

path_checks.to_csv(
    OUTPUTS["reports"] / "audited_manifest_path_check.csv",
    index=False
)

In [ ]:
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
imagenet_mean = weights.transforms().mean
imagenet_std = weights.transforms().std

train_transform = torch.nn.Sequential(
    # Placeholder to document the intended image preprocessing.
)

# Use PIL-compatible transforms from torchvision.
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.80, 1.00)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02,
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


class Main19Dataset(Dataset):
    def __init__(self, manifest_df, transform):
        self.df = manifest_df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["image_path"]

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as exc:
            raise RuntimeError(
                f"Could not load image at index {idx}: {image_path}"
            ) from exc

        image = self.transform(image)

        return {
            "image": image,
            "target": int(row["label_index"]),
            "image_id": row["image_id"],
        }


train_dataset = Main19Dataset(audited_train, train_transform)
calib_dataset = Main19Dataset(audited_calib, eval_transform)
test_dataset = Main19Dataset(audited_test, eval_transform)

print("Train dataset:", len(train_dataset))
print("Calibration dataset:", len(calib_dataset))
print("Audited test dataset:", len(test_dataset))

In [ ]:
generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
    generator=generator,
)

calib_loader = DataLoader(
    calib_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

train_targets = audited_train["label_index"].to_numpy()

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_targets,
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE
)

class_weight_table = pd.DataFrame(
    {
        "label_index": np.arange(NUM_CLASSES),
        "class_name": [index_to_class[i] for i in range(NUM_CLASSES)],
        "train_count": [
            int((audited_train["label_index"] == i).sum())
            for i in range(NUM_CLASSES)
        ],
        "class_weight": class_weights_np,
    }
)

class_weight_table.to_csv(
    OUTPUTS["reports"] / "class_weights.csv",
    index=False
)

display(class_weight_table)

In [ ]:
def build_efficientnet_b0(num_classes=NUM_CLASSES):
    model = models.efficientnet_b0(
        weights=EfficientNet_B0_Weights.IMAGENET1K_V1
    )
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


def multiclass_brier_score(probabilities, targets, num_classes=NUM_CLASSES):
    one_hot = np.eye(num_classes)[targets]
    return float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))


def expected_calibration_error(probabilities, targets, n_bins=ECE_BINS):
    confidences = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    correctness = (predictions == targets).astype(float)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    bin_records = []

    for lower, upper in zip(bin_edges[:-1], bin_edges[1:]):
        if upper == 1.0:
            mask = (confidences >= lower) & (confidences <= upper)
        else:
            mask = (confidences >= lower) & (confidences < upper)

        count = int(mask.sum())

        if count == 0:
            bin_records.append(
                {
                    "bin_lower": lower,
                    "bin_upper": upper,
                    "count": 0,
                    "mean_confidence": np.nan,
                    "accuracy": np.nan,
                    "gap": np.nan,
                }
            )
            continue

        mean_confidence = float(confidences[mask].mean())
        accuracy = float(correctness[mask].mean())
        gap = abs(accuracy - mean_confidence)
        ece += (count / len(targets)) * gap

        bin_records.append(
            {
                "bin_lower": lower,
                "bin_upper": upper,
                "count": count,
                "mean_confidence": mean_confidence,
                "accuracy": accuracy,
                "gap": gap,
            }
        )

    return float(ece), pd.DataFrame(bin_records)


def compute_metrics(probabilities, targets, class_names, prefix=""):
    predictions = probabilities.argmax(axis=1)

    accuracy = accuracy_score(targets, predictions)
    balanced_acc = balanced_accuracy_score(targets, predictions)

    precision, recall, f1, support = precision_recall_fscore_support(
        targets,
        predictions,
        labels=np.arange(NUM_CLASSES),
        zero_division=0,
    )

    _, _, macro_f1, _ = precision_recall_fscore_support(
        targets,
        predictions,
        average="macro",
        zero_division=0,
    )

    _, _, weighted_f1, _ = precision_recall_fscore_support(
        targets,
        predictions,
        average="weighted",
        zero_division=0,
    )

    nll = log_loss(
        targets,
        probabilities,
        labels=np.arange(NUM_CLASSES),
    )

    brier = multiclass_brier_score(probabilities, targets)
    ece, reliability_df = expected_calibration_error(probabilities, targets)

    top5 = top_k_accuracy_score(
        targets,
        probabilities,
        k=min(5, NUM_CLASSES),
        labels=np.arange(NUM_CLASSES),
    )

    metrics = {
        f"{prefix}accuracy": float(accuracy),
        f"{prefix}balanced_accuracy": float(balanced_acc),
        f"{prefix}macro_f1": float(macro_f1),
        f"{prefix}weighted_f1": float(weighted_f1),
        f"{prefix}nll": float(nll),
        f"{prefix}brier": float(brier),
        f"{prefix}ece_15_bins": float(ece),
        f"{prefix}top5_accuracy": float(top5),
    }

    per_class = pd.DataFrame(
        {
            "label_index": np.arange(NUM_CLASSES),
            "class_name": class_names,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
    )

    return metrics, per_class, reliability_df


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()

    all_logits = []
    all_targets = []
    all_ids = []

    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["target"].cpu().numpy()

        logits = model(images)

        all_logits.append(logits.cpu().numpy())
        all_targets.append(targets)
        all_ids.extend(batch["image_id"])

    logits = np.concatenate(all_logits, axis=0)
    targets = np.concatenate(all_targets, axis=0)

    return logits, targets, all_ids


def softmax_np(logits):
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["target"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        total += images.size(0)

    return {
        "loss": running_loss / total,
        "accuracy": correct / total,
    }


@torch.no_grad()
def evaluate_for_selection(model, loader):
    logits, targets, _ = predict_loader(model, loader)
    probabilities = softmax_np(logits)
    metrics, _, _ = compute_metrics(
        probabilities,
        targets,
        class_names,
    )
    return metrics

In [ ]:
seed_everything(SEED)

model = build_efficientnet_b0().to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[6, 13],
    gamma=0.5,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

checkpoint_path = OUTPUTS["checkpoints"] / "efficientnet_b0_audited_best.pt"

history = []
best_calib_macro_f1 = -np.inf
best_epoch = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()

    train_stats = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
    )

    calib_metrics_epoch = evaluate_for_selection(model, calib_loader)
    scheduler.step()

    row = {
        "epoch": epoch,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "train_loss": train_stats["loss"],
        "train_accuracy": train_stats["accuracy"],
        "calibration_accuracy": calib_metrics_epoch["accuracy"],
        "calibration_macro_f1": calib_metrics_epoch["macro_f1"],
        "calibration_nll": calib_metrics_epoch["nll"],
        "elapsed_seconds": time.time() - epoch_start,
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train loss={row['train_loss']:.5f} | "
        f"train acc={row['train_accuracy']:.4f} | "
        f"calib macro-F1={row['calibration_macro_f1']:.4f} | "
        f"calib NLL={row['calibration_nll']:.5f}"
    )

    if row["calibration_macro_f1"] > best_calib_macro_f1:
        best_calib_macro_f1 = row["calibration_macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_calibration_macro_f1": best_calib_macro_f1,
                "seed": SEED,
                "class_to_index": class_to_index,
                "audited_split_counts": audited_split_counts,
            },
            checkpoint_path,
        )
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping at epoch {epoch}; "
            f"best epoch was {best_epoch}."
        )
        break

history_df = pd.DataFrame(history)
history_df.to_csv(
    OUTPUTS["training"] / "efficientnet_b0_audited_training_history.csv",
    index=False
)

print("\nBest epoch:", best_epoch)
print("Best calibration macro-F1:", best_calib_macro_f1)
print("Checkpoint:", checkpoint_path)

assert checkpoint_path.exists()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o",
    label="Training loss",
)
axes[0].plot(
    history_df["epoch"],
    history_df["calibration_nll"],
    marker="o",
    label="Calibration NLL",
)
axes[0].axvline(
    best_epoch,
    color="black",
    linestyle="--",
    alpha=0.7,
    label=f"Selected epoch {best_epoch}",
)
axes[0].set_title("EfficientNet-B0 Training and Calibration Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(
    history_df["epoch"],
    history_df["train_accuracy"],
    marker="o",
    label="Training accuracy",
)
axes[1].plot(
    history_df["epoch"],
    history_df["calibration_macro_f1"],
    marker="o",
    label="Calibration macro-F1",
)
axes[1].axvline(
    best_epoch,
    color="black",
    linestyle="--",
    alpha=0.7,
    label=f"Selected epoch {best_epoch}",
)
axes[1].set_title("EfficientNet-B0 Training Progress")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()

plt.tight_layout()
fig.savefig(
    OUTPUTS["figures"] / "efficientnet_b0_audited_learning_curves.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False,
)

selected_model = build_efficientnet_b0().to(DEVICE)
selected_model.load_state_dict(checkpoint["model_state_dict"])
selected_model.eval()

assert checkpoint["epoch"] == best_epoch
assert checkpoint["audited_split_counts"] == audited_split_counts

calib_logits, calib_targets, calib_ids = predict_loader(
    selected_model,
    calib_loader
)

test_logits, test_targets, test_ids = predict_loader(
    selected_model,
    test_loader
)

assert len(calib_targets) == len(audited_calib) == 3197
assert len(test_targets) == len(audited_test) == 5320
assert set(test_ids) == set(audited_test["image_id"])

print("Calibration logits:", calib_logits.shape)
print("Audited test logits:", test_logits.shape)
print("Selected epoch:", checkpoint["epoch"])

In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        temperature = torch.exp(self.log_temperature)
        return logits / temperature

    @property
    def temperature(self):
        return float(torch.exp(self.log_temperature).detach().cpu().item())


temperature_scaler = TemperatureScaler().to(DEVICE)

calib_logits_tensor = torch.tensor(
    calib_logits,
    dtype=torch.float32,
    device=DEVICE,
)

calib_targets_tensor = torch.tensor(
    calib_targets,
    dtype=torch.long,
    device=DEVICE,
)

temperature_optimizer = torch.optim.LBFGS(
    temperature_scaler.parameters(),
    lr=0.01,
    max_iter=100,
    line_search_fn="strong_wolfe",
)

temperature_criterion = nn.CrossEntropyLoss()

def temperature_closure():
    temperature_optimizer.zero_grad()
    scaled_logits = temperature_scaler(calib_logits_tensor)
    loss = temperature_criterion(scaled_logits, calib_targets_tensor)
    loss.backward()
    return loss

temperature_optimizer.step(temperature_closure)

temperature_value = temperature_scaler.temperature

assert temperature_value > 0

with torch.no_grad():
    calibrated_test_logits = (
        temperature_scaler(
            torch.tensor(
                test_logits,
                dtype=torch.float32,
                device=DEVICE,
            )
        )
        .cpu()
        .numpy()
    )

print("Fitted temperature:", temperature_value)

temperature_metadata = {
    "temperature": temperature_value,
    "fitted_on": "audited_calibration_partition_only",
    "calibration_images": len(audited_calib),
    "audited_internal_test_images": len(audited_test),
    "selected_checkpoint_epoch": int(best_epoch),
    "seed": SEED,
}

with open(
    OUTPUTS["evaluation"] / "temperature_scaling_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(temperature_metadata, f, indent=2)

In [ ]:
test_prob_uncalibrated = softmax_np(test_logits)
test_prob_calibrated = softmax_np(calibrated_test_logits)

uncalibrated_metrics, uncalibrated_per_class, reliability_uncalibrated = compute_metrics(
    test_prob_uncalibrated,
    test_targets,
    class_names,
    prefix="uncalibrated_",
)

calibrated_metrics, calibrated_per_class, reliability_calibrated = compute_metrics(
    test_prob_calibrated,
    test_targets,
    class_names,
    prefix="temperature_scaled_",
)

final_metrics = {
    "model": "EfficientNet-B0",
    "selected_checkpoint_epoch": int(best_epoch),
    "temperature": temperature_value,
    "audit_candidate_pairs": len(main19_reviews),
    "audit_confirmed_leakage_pairs": len(confirmed_main19),
    "audit_unresolved_pairs": len(unresolved_main19),
    "audited_train_images": len(audited_train),
    "audited_calibration_images": len(audited_calib),
    "audited_internal_test_images": len(audited_test),
    **uncalibrated_metrics,
    **calibrated_metrics,
}

final_metrics_df = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in final_metrics.items()]
)

display(final_metrics_df)

final_metrics_df.to_csv(
    OUTPUTS["evaluation"] / "efficientnet_b0_audited_final_metrics.csv",
    index=False
)

uncalibrated_per_class.to_csv(
    OUTPUTS["evaluation"] / "efficientnet_b0_audited_per_class_uncalibrated.csv",
    index=False
)

calibrated_per_class.to_csv(
    OUTPUTS["evaluation"] / "efficientnet_b0_audited_per_class_temperature_scaled.csv",
    index=False
)

reliability_uncalibrated.to_csv(
    OUTPUTS["evaluation"] / "reliability_uncalibrated.csv",
    index=False
)

reliability_calibrated.to_csv(
    OUTPUTS["evaluation"] / "reliability_temperature_scaled.csv",
    index=False
)

In [ ]:
final_predictions = pd.DataFrame(
    {
        "image_id": test_ids,
        "true_label_index": test_targets,
        "true_label": [index_to_class[int(i)] for i in test_targets],
        "predicted_label_index": test_prob_calibrated.argmax(axis=1),
        "predicted_label": [
            index_to_class[int(i)]
            for i in test_prob_calibrated.argmax(axis=1)
        ],
        "confidence_uncalibrated": test_prob_uncalibrated.max(axis=1),
        "confidence_temperature_scaled": test_prob_calibrated.max(axis=1),
        "correct": (
            test_prob_calibrated.argmax(axis=1) == test_targets
        ).astype(int),
    }
)

for class_index, class_name in index_to_class.items():
    final_predictions[f"prob_{class_index:02d}_{class_name}"] = (
        test_prob_calibrated[:, class_index]
    )

final_predictions.to_csv(
    OUTPUTS["evaluation"] / "efficientnet_b0_audited_test_predictions.csv",
    index=False
)

cm = confusion_matrix(
    test_targets,
    test_prob_calibrated.argmax(axis=1),
    labels=np.arange(NUM_CLASSES),
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names,
)

cm_df.to_csv(
    OUTPUTS["evaluation"] / "efficientnet_b0_audited_confusion_matrix.csv"
)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm_df,
    cmap="Blues",
    square=True,
    cbar_kws={"shrink": 0.75},
    ax=ax,
)
ax.set_title("EfficientNet-B0 Confusion Matrix on Audited Main19 Internal Test")
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()

fig.savefig(
    OUTPUTS["figures"] / "efficientnet_b0_audited_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

for ax, reliability_df, title in [
    (
        axes[0],
        reliability_uncalibrated,
        "Uncalibrated",
    ),
    (
        axes[1],
        reliability_calibrated,
        f"Temperature-scaled (T={temperature_value:.4f})",
    ),
]:
    plot_df = reliability_df.dropna(subset=["mean_confidence", "accuracy"])

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="black",
        label="Perfect calibration",
    )

    ax.bar(
        plot_df["mean_confidence"],
        plot_df["accuracy"],
        width=1 / ECE_BINS,
        alpha=0.7,
        color="#4C78A8",
        edgecolor="white",
        label="Observed accuracy",
    )

    ax.set_title(title)
    ax.set_xlabel("Mean confidence")
    ax.set_ylabel("Accuracy")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()

plt.tight_layout()
fig.savefig(
    OUTPUTS["figures"] / "efficientnet_b0_audited_reliability_diagrams.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
def selective_metrics(probabilities, targets, threshold):
    confidences = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    accepted = confidences >= threshold

    coverage = float(accepted.mean())
    rejected = int((~accepted).sum())

    if accepted.sum() == 0:
        return {
            "threshold": threshold,
            "coverage": coverage,
            "rejected": rejected,
            "accepted": 0,
            "selective_accuracy": np.nan,
            "selective_risk": np.nan,
        }

    selective_accuracy = float(
        accuracy_score(targets[accepted], predictions[accepted])
    )

    return {
        "threshold": threshold,
        "coverage": coverage,
        "rejected": rejected,
        "accepted": int(accepted.sum()),
        "selective_accuracy": selective_accuracy,
        "selective_risk": float(1.0 - selective_accuracy),
    }


calib_prob_for_selective = softmax_np(calib_logits)
test_prob_for_selective = test_prob_uncalibrated

threshold_grid = np.round(np.arange(0.50, 1.001, 0.01), 2)

calib_selective_rows = [
    selective_metrics(calib_prob_for_selective, calib_targets, threshold)
    for threshold in threshold_grid
]

calib_selective_df = pd.DataFrame(calib_selective_rows)

eligible_thresholds = calib_selective_df.loc[
    calib_selective_df["selective_risk"].le(0.01)
    & calib_selective_df["coverage"].gt(0)
].copy()

if len(eligible_thresholds) > 0:
    selected_threshold = float(
        eligible_thresholds
        .sort_values(
            ["coverage", "threshold"],
            ascending=[False, True],
        )
        .iloc[0]["threshold"]
    )
else:
    selected_threshold = 1.00

test_selective_result = selective_metrics(
    test_prob_for_selective,
    test_targets,
    selected_threshold,
)

calib_selective_df.to_csv(
    OUTPUTS["evaluation"] / "calibration_selective_prediction_curve.csv",
    index=False
)

pd.DataFrame([test_selective_result]).to_csv(
    OUTPUTS["evaluation"] / "audited_test_selective_prediction_result.csv",
    index=False
)

print("Selected threshold from calibration only:", selected_threshold)
display(pd.DataFrame([test_selective_result]))

In [ ]:
test_risk_coverage_rows = [
    selective_metrics(test_prob_for_selective, test_targets, threshold)
    for threshold in threshold_grid
]

test_risk_coverage_df = pd.DataFrame(test_risk_coverage_rows)

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(
    test_risk_coverage_df["coverage"],
    test_risk_coverage_df["selective_risk"],
    marker="o",
    markersize=3,
    color="#E45756",
    label="Audited internal test",
)

selected_row = test_risk_coverage_df.loc[
    np.isclose(
        test_risk_coverage_df["threshold"],
        selected_threshold
    )
].iloc[0]

ax.scatter(
    selected_row["coverage"],
    selected_row["selective_risk"],
    s=90,
    color="black",
    zorder=3,
    label=f"Selected threshold = {selected_threshold:.2f}",
)

ax.set_title("Risk-Coverage Curve on Audited Main19 Test Set")
ax.set_xlabel("Coverage")
ax.set_ylabel("Selective risk")
ax.set_xlim(0, 1.02)
ax.set_ylim(bottom=0)
ax.legend()

plt.tight_layout()
fig.savefig(
    OUTPUTS["figures"] / "efficientnet_b0_audited_risk_coverage_curve.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

test_risk_coverage_df.to_csv(
    OUTPUTS["evaluation"] / "audited_test_risk_coverage_curve.csv",
    index=False
)

In [ ]:
def bootstrap_metric_ci(
    probabilities,
    targets,
    metric_name,
    n_bootstrap=2000,
    seed=SEED,
):
    rng = np.random.default_rng(seed)
    n = len(targets)
    predictions = probabilities.argmax(axis=1)

    scores = []

    for _ in range(n_bootstrap):
        indices = rng.integers(0, n, size=n)
        y_true = targets[indices]
        y_pred = predictions[indices]

        if metric_name == "accuracy":
            score = accuracy_score(y_true, y_pred)

        elif metric_name == "macro_f1":
            score = precision_recall_fscore_support(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )[2]

        elif metric_name == "balanced_accuracy":
            score = balanced_accuracy_score(y_true, y_pred)

        else:
            raise ValueError(f"Unsupported metric: {metric_name}")

        scores.append(score)

    lower, upper = np.percentile(scores, [2.5, 97.5])

    return {
        "metric": metric_name,
        "estimate": float(
            accuracy_score(targets, predictions)
            if metric_name == "accuracy"
            else precision_recall_fscore_support(
                targets,
                predictions,
                average="macro",
                zero_division=0,
            )[2]
            if metric_name == "macro_f1"
            else balanced_accuracy_score(targets, predictions)
        ),
        "ci_95_lower": float(lower),
        "ci_95_upper": float(upper),
        "bootstrap_iterations": n_bootstrap,
    }


bootstrap_results = pd.DataFrame(
    [
        bootstrap_metric_ci(
            test_prob_calibrated,
            test_targets,
            metric_name="accuracy",
        ),
        bootstrap_metric_ci(
            test_prob_calibrated,
            test_targets,
            metric_name="macro_f1",
        ),
        bootstrap_metric_ci(
            test_prob_calibrated,
            test_targets,
            metric_name="balanced_accuracy",
        ),
    ]
)

bootstrap_results.to_csv(
    OUTPUTS["evaluation"] / "audited_test_bootstrap_confidence_intervals.csv",
    index=False
)

display(bootstrap_results)

In [ ]:
error_analysis = final_predictions.copy()

error_summary = pd.DataFrame(
    {
        "metric": [
            "test_images",
            "correct_predictions",
            "incorrect_predictions",
            "mean_confidence_correct",
            "mean_confidence_incorrect",
            "median_confidence_correct",
            "median_confidence_incorrect",
        ],
        "value": [
            len(error_analysis),
            int(error_analysis["correct"].sum()),
            int((1 - error_analysis["correct"]).sum()),
            float(
                error_analysis.loc[
                    error_analysis["correct"].eq(1),
                    "confidence_temperature_scaled",
                ].mean()
            ),
            float(
                error_analysis.loc[
                    error_analysis["correct"].eq(0),
                    "confidence_temperature_scaled",
                ].mean()
            ),
            float(
                error_analysis.loc[
                    error_analysis["correct"].eq(1),
                    "confidence_temperature_scaled",
                ].median()
            ),
            float(
                error_analysis.loc[
                    error_analysis["correct"].eq(0),
                    "confidence_temperature_scaled",
                ].median()
            ),
        ],
    }
)

error_summary.to_csv(
    OUTPUTS["evaluation"] / "audited_test_error_confidence_summary.csv",
    index=False
)

top_errors = (
    error_analysis.loc[error_analysis["correct"].eq(0)]
    .sort_values("confidence_temperature_scaled", ascending=False)
    .head(50)
)

top_errors.to_csv(
    OUTPUTS["evaluation"] / "top_50_high_confidence_errors.csv",
    index=False
)

display(error_summary)
display(
    top_errors[
        [
            "image_id",
            "true_label",
            "predicted_label",
            "confidence_temperature_scaled",
        ]
    ].head(15)
)

In [ ]:
def sha256_file(path):
    sha256 = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(block)
    return sha256.hexdigest()


artifact_paths = [
    OUTPUTS["audit"] / "main19_duplicate_audit_summary.csv",
    OUTPUTS["audit"] / "main19_confirmed_leakage_pairs.csv",
    OUTPUTS["audit"] / "main19_unresolved_near_duplicate_pairs.csv",
    OUTPUTS["manifests"] / "main19_audited_train.csv",
    OUTPUTS["manifests"] / "main19_audited_calibration.csv",
    OUTPUTS["manifests"] / "main19_audited_internal_test.csv",
    checkpoint_path,
    OUTPUTS["evaluation"] / "efficientnet_b0_audited_final_metrics.csv",
    OUTPUTS["evaluation"] / "efficientnet_b0_audited_test_predictions.csv",
]

artifact_hashes = {
    str(path.relative_to(WORK_ROOT)): sha256_file(path)
    for path in artifact_paths
    if path.exists()
}

reproducibility_manifest = {
    "notebook_purpose": "Leakage-aware Main19 EfficientNet-B0 retraining and audited evaluation",
    "seed": SEED,
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "model": "EfficientNet-B0",
    "pretrained_weights": "EfficientNet_B0_Weights.IMAGENET1K_V1",
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "maximum_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "checkpoint_selection_metric": "calibration_macro_f1",
    "temperature_fit_partition": "audited_calibration_only",
    "test_partition": "audited_internal_test_only",
    "audit": {
        "main19_candidate_pairs": len(main19_reviews),
        "confirmed_leakage_pairs": len(confirmed_main19),
        "unresolved_pairs": len(unresolved_main19),
        "excluded_images": len(excluded_confirmed_rows),
        "audited_split_counts": audited_split_counts,
    },
    "artifact_sha256": artifact_hashes,
}

with open(
    OUTPUTS["reports"] / "reproducibility_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(reproducibility_manifest, f, indent=2)

print(json.dumps(reproducibility_manifest, indent=2))

In [ ]:
metric_lookup = dict(
    zip(final_metrics_df["metric"], final_metrics_df["value"])
)

ci_lookup = bootstrap_results.set_index("metric").to_dict("index")

paper_summary = f"""
AUDITED MAIN19 EVALUATION SUMMARY

Dataset audit:
- Main19 candidate near-duplicate pairs: {len(main19_reviews)}
- Completed manual decisions: {len(completed_main19)}
- Confirmed cross-split leakage pairs: {len(confirmed_main19)}
- Unresolved candidate pairs: {len(unresolved_main19)}
- Images excluded from internal test: {len(excluded_confirmed_rows)}
- Audited split counts: train={len(audited_train)}, calibration={len(audited_calib)}, internal_test={len(audited_test)}

Model protocol:
- Model: EfficientNet-B0, ImageNet-pretrained
- Checkpoint selected by calibration macro-F1 only
- Selected epoch: {best_epoch}
- Temperature fitted on calibration partition only: T={temperature_value:.4f}

Audited internal-test performance:
- Uncalibrated accuracy: {metric_lookup['uncalibrated_accuracy']:.4f}
- Uncalibrated macro-F1: {metric_lookup['uncalibrated_macro_f1']:.4f}
- Temperature-scaled accuracy: {metric_lookup['temperature_scaled_accuracy']:.4f}
- Temperature-scaled macro-F1: {metric_lookup['temperature_scaled_macro_f1']:.4f}
- Temperature-scaled balanced accuracy: {metric_lookup['temperature_scaled_balanced_accuracy']:.4f}
- Temperature-scaled NLL: {metric_lookup['temperature_scaled_nll']:.5f}
- Temperature-scaled Brier score: {metric_lookup['temperature_scaled_brier']:.5f}
- Temperature-scaled ECE (15 bins): {metric_lookup['temperature_scaled_ece_15_bins']:.5f}
- Top-5 accuracy: {metric_lookup['temperature_scaled_top5_accuracy']:.4f}

Bootstrap 95% confidence intervals:
- Accuracy: {ci_lookup['accuracy']['estimate']:.4f}
  [{ci_lookup['accuracy']['ci_95_lower']:.4f}, {ci_lookup['accuracy']['ci_95_upper']:.4f}]
- Macro-F1: {ci_lookup['macro_f1']['estimate']:.4f}
  [{ci_lookup['macro_f1']['ci_95_lower']:.4f}, {ci_lookup['macro_f1']['ci_95_upper']:.4f}]
- Balanced accuracy: {ci_lookup['balanced_accuracy']['estimate']:.4f}
  [{ci_lookup['balanced_accuracy']['ci_95_lower']:.4f}, {ci_lookup['balanced_accuracy']['ci_95_upper']:.4f}]

Selective prediction:
- Threshold selected using calibration only: {selected_threshold:.2f}
- Audited test coverage: {test_selective_result['coverage']:.4f}
- Audited test selective accuracy: {test_selective_result['selective_accuracy']:.4f}
- Audited test selective risk: {test_selective_result['selective_risk']:.4f}

Interpretation:
The duplicate audit reduces documented cross-split leakage by removing two manually
confirmed validation-side duplicates. Nine pHash candidate pairs remain unresolved,
so the audited benchmark should not be described as fully leakage-free.
"""

summary_path = OUTPUTS["reports"] / "paper_ready_audited_summary.txt"
summary_path.write_text(paper_summary, encoding="utf-8")

print(paper_summary)

In [ ]:
required_outputs = [
    OUTPUTS["audit"] / "main19_duplicate_audit_summary.csv",
    OUTPUTS["audit"] / "main19_confirmed_leakage_pairs.csv",
    OUTPUTS["audit"] / "main19_unresolved_near_duplicate_pairs.csv",
    OUTPUTS["manifests"] / "main19_audited_train.csv",
    OUTPUTS["manifests"] / "main19_audited_calibration.csv",
    OUTPUTS["manifests"] / "main19_audited_internal_test.csv",
    checkpoint_path,
    OUTPUTS["evaluation"] / "efficientnet_b0_audited_final_metrics.csv",
    OUTPUTS["evaluation"] / "efficientnet_b0_audited_test_predictions.csv",
    OUTPUTS["evaluation"] / "audited_test_bootstrap_confidence_intervals.csv",
    OUTPUTS["figures"] / "main19_duplicate_audit_and_split_counts.png",
    OUTPUTS["figures"] / "efficientnet_b0_audited_learning_curves.png",
    OUTPUTS["figures"] / "efficientnet_b0_audited_confusion_matrix.png",
    OUTPUTS["figures"] / "efficientnet_b0_audited_reliability_diagrams.png",
    OUTPUTS["figures"] / "efficientnet_b0_audited_risk_coverage_curve.png",
    OUTPUTS["reports"] / "reproducibility_manifest.json",
    OUTPUTS["reports"] / "paper_ready_audited_summary.txt",
]

missing_outputs = [str(path) for path in required_outputs if not path.exists()]

assert not missing_outputs, (
    "Notebook completed with missing required outputs:\n"
    + "\n".join(missing_outputs)
)

assert len(audited_train) == 18113
assert len(audited_calib) == 3197
assert len(audited_test) == 5320
assert len(confirmed_main19) == 2
assert len(unresolved_main19) == 9
assert len(final_predictions) == 5320
assert final_predictions["image_id"].nunique() == 5320
assert not final_predictions["image_id"].isin(exclude_image_ids).any()

print("FINAL INTEGRITY GATE PASSED")
print("All audit, training, evaluation, figure, and reproducibility artifacts exist.")
print("Working output directory:", WORK_ROOT)